In [ ]:
import pickle
from pathlib import Path

def load_inference_cache(load_path='inference_cache_base.pkl'):
    if Path(load_path).exists():
        with open(load_path, 'rb') as f:
            cache = pickle.load(f)
        print(f"Cache loaded from {load_path}")
        return cache
    else:
        print(f"Cache file not found: {load_path}")
        return None


In [ ]:
inference_cache_lambda_005 = load_inference_cache('inference_lambda0.005_cache.pkl')
inference_cache_lambda_01 = load_inference_cache('inference_lambda0.01_cache.pkl')
inference_cache_lambda_05 = load_inference_cache('inference_lambda0.05_cache.pkl')

path_lambda_005 = "/models/jhkim/imagenet200_lambda0.005"
path_lambda_01 = "/models/jhkim/imagenet200_lambda0.01"
path_lambda_05 = "/models/jhkim/imagenet200_lambda0.05"

In [ ]:
import torch
from torchvision.datasets import ImageNet
import torch.nn as nn
from torchvision import transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import xml.etree.ElementTree as ET
from PIL import Image, ImageDraw
import os
from torch.utils.data import Dataset, DataLoader, Subset
import pandas as pd


device = torch.device("cuda")

In [ ]:
sns.set_theme(style="whitegrid", font="cmr10", font_scale=1.9)
plt.rcParams["axes.formatter.use_mathtext"] = True
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42})
colors_darker = sns.color_palette("dark", 5 * 2)
colors = sns.color_palette("deep", 5 * 2)
colors

In [ ]:
class LinearClassifier(nn.Module):
    """Linear layer to train on top of frozen features"""

    def __init__(self, dim, num_labels=1000):
        super(LinearClassifier, self).__init__()
        self.num_labels = num_labels
        self.linear = nn.Linear(dim, num_labels)
        self.linear.weight.data.normal_(mean=0.0, std=0.01)
        self.linear.bias.data.zero_()

    def forward(self, x):
        # flatten
        x = x.view(x.size(0), -1)

        # linear layer
        return self.linear(x)

In [ ]:
def get_bbox_area_ratio_after_crop(xml_path, original_width, original_height, crop_size=224):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    if original_width < original_height:
        resize_width = 256
        resize_height = int(original_height * 256 / original_width)
    else:
        resize_height = 256
        resize_width = int(original_width * 256 / original_height)

    crop_left = (resize_width - crop_size) // 2
    crop_top = (resize_height - crop_size) // 2
    crop_right = crop_left + crop_size
    crop_bottom = crop_top + crop_size

    mask = np.zeros((crop_size, crop_size), dtype=bool)

    for obj in root.findall('object'):
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)

        scale_x = resize_width / original_width
        scale_y = resize_height / original_height

        xmin_resized = xmin * scale_x
        ymin_resized = ymin * scale_y
        xmax_resized = xmax * scale_x
        ymax_resized = ymax * scale_y

        intersect_xmin = max(xmin_resized, crop_left)
        intersect_ymin = max(ymin_resized, crop_top)
        intersect_xmax = min(xmax_resized, crop_right)
        intersect_ymax = min(ymax_resized, crop_bottom)

        if intersect_xmin < intersect_xmax and intersect_ymin < intersect_ymax:
            x1 = int(intersect_xmin - crop_left)
            y1 = int(intersect_ymin - crop_top)
            x2 = int(intersect_xmax - crop_left)
            y2 = int(intersect_ymax - crop_top)

            mask[y1:y2, x1:x2] = True

    object_pixels = np.sum(mask)
    crop_area = crop_size * crop_size

    return object_pixels / crop_area

In [ ]:
def load_ddp_checkpoint(model, checkpoint_path):
    state_dict = torch.load(checkpoint_path)
    new_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(new_state_dict)
    return model

def parse_bbox_annotation(xml_path):
    if not os.path.exists(xml_path):
        return []

    tree = ET.parse(xml_path)
    root = tree.getroot()

    bboxes = []
    for obj in root.findall('object'):
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)

        bboxes.append([xmin, ymin, xmax, ymax])

    return bboxes

def mask_object_regions(image_path, xml_path):
    img = Image.open(image_path).convert('RGB')

    bboxes = parse_bbox_annotation(xml_path)

    if bboxes:
        draw = ImageDraw.Draw(img)
        for bbox in bboxes:
            xmin, ymin, xmax, ymax = bbox
            draw.rectangle([xmin, ymin, xmax, ymax], fill='black')

    return img

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

def denormalize(tensor):
    mean = torch.tensor((0.485, 0.456, 0.406)).view(3, 1, 1).to(tensor.device)
    std = torch.tensor((0.229, 0.224, 0.225)).view(3, 1, 1).to(tensor.device)
    denorm = tensor * std + mean
    return torch.clamp(denorm, 0, 1)

dataset_test = ImageNet("/home/dataset/imagenet", split="val", transform=val_transform)
bbox_dir = "/home/dataset/imagenet/ILSVRC/Annotations/CLS-LOC/val"

In [ ]:
def load_model(model, path):
    if os.path.isfile(path):
        datas = torch.load(path, weights_only=False)
        if 'model' in datas:
            state_dict = {k.replace('module.', '').replace('backbone.', ''): v
                          for k, v in datas['model'].items()
                          if k.replace('module.', '').startswith('backbone.')}
            model.load_state_dict(state_dict, strict=False)
        else:
            model.load_state_dict(datas)
        print(f"loaded state dict {path}")

In [ ]:
steps_int = [5*i for i in range(0, 11)]
steps = [str(i) for i in steps_int]

In [ ]:
class_data = {}
all_data_list = []

print("Collecting images with bounding boxes for all classes...")
for idx in tqdm(range(len(dataset_test))):
    image_path, label = dataset_test.samples[idx]

    if label not in class_data:
        class_data[label] = []

    img_name = os.path.basename(image_path)
    xml_name = os.path.splitext(img_name)[0] + ".xml"
    xml_path = os.path.join(bbox_dir, xml_name)

    if os.path.exists(xml_path):
        img = Image.open(image_path)
        img_width, img_height = img.size

        bbox_ratio = get_bbox_area_ratio_after_crop(xml_path, img_width, img_height, crop_size=224)

        class_data[label].append({
            'idx': idx,
            'image_path': image_path,
            'xml_path': xml_path,
            'bbox_ratio': bbox_ratio
        })
        all_data_list.append({
            'image_path': image_path,
            'xml_path': xml_path,
            'class': label,
        })

valid_classes = {cls: data for cls, data in class_data.items() if len(data) >= 10}
print(f"Found {len(valid_classes)} classes with at least 10 images with bboxes")

class_bbox_stats = {}
for cls, data in valid_classes.items():
    bbox_ratios = [d['bbox_ratio'] for d in data]
    class_bbox_stats[cls] = {
        'mean_bbox_ratio': np.mean(bbox_ratios),
        'n_samples': len(data)
    }

bbox_sizes = [class_bbox_stats[cls]['mean_bbox_ratio'] for cls in valid_classes.keys()]
bbox_percentiles = np.percentile(bbox_sizes, [5, 10, 15, 20])

class_bbox_group = {}
for cls in valid_classes.keys():
    bbox_ratio = class_bbox_stats[cls]['mean_bbox_ratio']
    if bbox_percentiles[0] <= bbox_ratio < bbox_percentiles[1]:
        class_bbox_group[cls] = 'Small'
    elif bbox_percentiles[1] <= bbox_ratio < bbox_percentiles[2]:
        class_bbox_group[cls] = 'Medium'
    elif bbox_percentiles[2] <= bbox_ratio < bbox_percentiles[3]:
        class_bbox_group[cls] = 'Large'
    else:
        class_bbox_group[cls] = 'Other'

all_images = []
for cls, data_list in valid_classes.items():
    for d in data_list:
        all_images.append({
            'idx': d['idx'],
            'class': cls,
            'bbox_ratio': d['bbox_ratio']
        })

In [ ]:
class MaskedImageDataset(Dataset):
    def __init__(self, data_list, transform):
        self.data_list = data_list
        self.transform = transform
        self.preprocess()

    def preprocess(self):
        print("Preprocessing bboxes...")
        for data in tqdm(self.data_list):
            img = Image.open(data['image_path'])
            orig_w, orig_h = img.size

            bboxes = parse_bbox_annotation(data['xml_path'])

            data['orig_size'] = (orig_w, orig_h)
            data['bboxes'] = bboxes

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        data = self.data_list[idx]

        img = Image.open(data['image_path']).convert('RGB')
        orig_w, orig_h = data['orig_size']
        bboxes = data['bboxes']

        # Apply transform to image
        img_tensor = self.transform(img)
        H, W = img_tensor.shape[1], img_tensor.shape[2]  # 224, 224

        # Create bbox mask on PIL Image, then transform it
        bbox_mask_pil = Image.new('L', (orig_w, orig_h), 0)
        draw = ImageDraw.Draw(bbox_mask_pil)

        for bbox in bboxes:
            xmin, ymin, xmax, ymax = bbox
            draw.rectangle([xmin, ymin, xmax, ymax], fill=255)

        # Apply same transform to mask (without normalization)
        mask_transform = transforms.Compose([
            transforms.Resize(256, interpolation=3),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
        ])

        bbox_mask_tensor = mask_transform(bbox_mask_pil)  # (1, 224, 224)
        bbox_mask_tensor = (bbox_mask_tensor > 0.5).float()  # Binarize

        # Create 3-channel mask
        mask = 1 - bbox_mask_tensor.repeat(3, 1, 1)  # 0 where bbox, 1 elsewhere

        object_tensor = img_tensor * (1 - mask)
        background_tensor = img_tensor * mask

        return img_tensor, object_tensor, background_tensor, data['class'], mask, idx

In [ ]:
results_lambda_005 = {step: {} for step in steps}
results_lambda_01 = {step: {} for step in steps}
results_lambda_05 = {step: {} for step in steps}

print("Calculating global class statistics...")
for cache, results in [
    (inference_cache_lambda_005, results_lambda_005),
    (inference_cache_lambda_01, results_lambda_01),
    (inference_cache_lambda_05, results_lambda_05),
]:
    for step in steps:
        class_stats = {cls: {'orig_top1': 0, 'orig_top5': 0,
                             'object_top1': 0, 'object_top5': 0,
                             'background_top1': 0, 'background_top5': 0,
                             'total': 0}
                       for cls in valid_classes.keys()}

        step_data = cache[step]

        for idx, data in step_data.items():
            cls = data['class']
            if cls not in class_stats: continue

            class_stats[cls]['orig_top1'] += int(data['orig_top1'])
            class_stats[cls]['orig_top5'] += int(data['orig_top5'])
            class_stats[cls]['object_top1'] += int(data['obj_top1'])
            class_stats[cls]['object_top5'] += int(data['obj_top5'])
            class_stats[cls]['background_top1'] += int(data['bg_top1'])
            class_stats[cls]['background_top5'] += int(data['bg_top5'])
            class_stats[cls]['total'] += 1

        for cls, stats in class_stats.items():
            if stats['total'] > 0:
                results[step][cls] = {
                    k: v / stats['total'] for k, v in stats.items() if k != 'total'
                }
                results[step][cls]['mean_bbox_ratio'] = class_bbox_stats[cls]['mean_bbox_ratio']

In [ ]:
target_epochs = range(len(steps))
epoch_names = {k: steps_int[k] for k in target_epochs}

def build_sample_df(cache):
    plot_data = []
    for epoch_idx in target_epochs:
        try:
            step_key = steps[epoch_idx]
        except (IndexError, KeyError):
            continue
        step_data = cache[step_key]
        step_value = epoch_names.get(epoch_idx, epoch_idx)
        for idx, data in step_data.items():
            cls = data['class']
            if cls not in class_data:
                continue
            sample_info = None
            for sample in class_data[cls]:
                if sample['idx'] == idx:
                    sample_info = sample
                    break
            bbox_ratio = sample_info['bbox_ratio']
            if bbox_ratio == 0.0:
                continue
            bg_acc = int(data['bg_top5'])
            obj_acc = int(data['obj_top5'])
            diff_5 = (bg_acc - obj_acc) * 100
            plot_data.append({
                'Step': step_value,
                'Class': cls,
                'Sample_Idx': idx,
                'Diff_Top5': diff_5,
                'BBox Ratio': bbox_ratio
            })
    return pd.DataFrame(plot_data)

df_lambda_005 = build_sample_df(inference_cache_lambda_005)
df_lambda_01 = build_sample_df(inference_cache_lambda_01)
df_lambda_05 = build_sample_df(inference_cache_lambda_05)

n_bins = 25
all_bbox = pd.concat([df_lambda_005['BBox Ratio'], df_lambda_01['BBox Ratio'], df_lambda_05['BBox Ratio']])
_, bin_edges = pd.cut(all_bbox, bins=n_bins, retbins=True)

df_lambda_005['BBox Bin'] = pd.cut(df_lambda_005['BBox Ratio'], bins=bin_edges, labels=False)
df_lambda_01['BBox Bin'] = pd.cut(df_lambda_01['BBox Ratio'], bins=bin_edges, labels=False)
df_lambda_05['BBox Bin'] = pd.cut(df_lambda_05['BBox Ratio'], bins=bin_edges, labels=False)

pivot_lambda_005 = df_lambda_005.pivot_table(values='Diff_Top5', index='BBox Bin', columns='Step', aggfunc='mean')
pivot_lambda_01 = df_lambda_01.pivot_table(values='Diff_Top5', index='BBox Bin', columns='Step', aggfunc='mean')
pivot_lambda_05 = df_lambda_05.pivot_table(values='Diff_Top5', index='BBox Bin', columns='Step', aggfunc='mean')

step_values = sorted(set(pivot_lambda_005.columns) | set(pivot_lambda_01.columns) | set(pivot_lambda_05.columns))
pivot_lambda_005 = pivot_lambda_005.reindex(columns=step_values)
pivot_lambda_01 = pivot_lambda_01.reindex(columns=step_values)
pivot_lambda_05 = pivot_lambda_05.reindex(columns=step_values)

x_edges = np.concatenate([[0], step_values])
y_edges = bin_edges

fig, axes = plt.subplots(1, 3, figsize=(42, 13), sharey=True, constrained_layout=True)

vmin, vmax = -80, 40

im0 = axes[0].pcolormesh(x_edges, y_edges, pivot_lambda_005.values, cmap='RdBu_r', vmin=vmin, vmax=vmax)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Object Ratio')
axes[0].set_title(r'$\lambda=0.005$')

im1 = axes[1].pcolormesh(x_edges, y_edges, pivot_lambda_01.values, cmap='RdBu_r', vmin=vmin, vmax=vmax)
axes[1].set_xlabel('Step')
axes[1].set_title(r'$\lambda=0.01$')

im2 = axes[2].pcolormesh(x_edges, y_edges, pivot_lambda_05.values, cmap='RdBu_r', vmin=vmin, vmax=vmax)
axes[2].set_xlabel('Step')
axes[2].set_title(r'$\lambda=0.05$')
plt.colorbar(im2, ax=axes, label='Top-5 Spuriosity')

plt.show()

print(f"Lambda 0.005 - Total data points: {len(df_lambda_005)} (= ~50k samples × {len(step_values)} steps)")
print(f"Lambda 0.01  - Total data points: {len(df_lambda_01)} (= ~50k samples × {len(step_values)} steps)")
print(f"Lambda 0.05  - Total data points: {len(df_lambda_05)} (= ~50k samples × {len(step_values)} steps)")
print(f"BBox Ratio range: {all_bbox.min():.4f} ~ {all_bbox.max():.4f}")

In [ ]:
# ===== Choose two lambda configs to compare =====
# Options: inference_cache_lambda_005, inference_cache_lambda_01, inference_cache_lambda_05
cache_base = inference_cache_lambda_01
cache_lambda = inference_cache_lambda_05
label_base = r"$\lambda=0.005$"
label_lambda = r"$\lambda=0.05$"

results_base = {step: {} for step in steps}
results_lambda = {step: {} for step in steps}

print("Calculating global class statistics...")
for cache, results in [(cache_base, results_base), (cache_lambda, results_lambda)]:
    for step in steps:
        class_stats = {cls: {'orig_top1': 0, 'orig_top5': 0,
                             'object_top1': 0, 'object_top5': 0,
                             'background_top1': 0, 'background_top5': 0,
                             'total': 0}
                       for cls in valid_classes.keys()}

        step_data = cache[step]

        for idx, data in step_data.items():
            cls = data['class']
            if cls not in class_stats: continue

            class_stats[cls]['orig_top1'] += int(data['orig_top1'])
            class_stats[cls]['orig_top5'] += int(data['orig_top5'])
            class_stats[cls]['object_top1'] += int(data['obj_top1'])
            class_stats[cls]['object_top5'] += int(data['obj_top5'])
            class_stats[cls]['background_top1'] += int(data['bg_top1'])
            class_stats[cls]['background_top5'] += int(data['bg_top5'])
            class_stats[cls]['total'] += 1

        for cls, stats in class_stats.items():
            if stats['total'] > 0:
                results[step][cls] = {
                    k: v / stats['total'] for k, v in stats.items() if k != 'total'
                }
                results[step][cls]['mean_bbox_ratio'] = class_bbox_stats[cls]['mean_bbox_ratio']

In [ ]:
target_epochs = range(len(steps))
epoch_names = {k: steps_int[k] for k in target_epochs}

def build_sample_df(cache):
    plot_data = []
    for epoch_idx in target_epochs:
        try:
            step_key = steps[epoch_idx]
        except (IndexError, KeyError):
            continue
        step_data = cache[step_key]
        step_value = epoch_names.get(epoch_idx, epoch_idx)
        for idx, data in step_data.items():
            cls = data['class']
            if cls not in class_data:
                continue
            sample_info = None
            for sample in class_data[cls]:
                if sample['idx'] == idx:
                    sample_info = sample
                    break
            bbox_ratio = sample_info['bbox_ratio']
            if bbox_ratio == 0.0:
                continue
            bg_acc = int(data['bg_top1'])
            obj_acc = int(data['obj_top1'])
            diff_5 = (bg_acc - obj_acc) * 100
            plot_data.append({
                'Step': step_value,
                'Class': cls,
                'Sample_Idx': idx,
                'Diff_Top5': diff_5,
                'BBox Ratio': bbox_ratio
            })
    return pd.DataFrame(plot_data)

df_base = build_sample_df(cache_base)
df_lambda = build_sample_df(cache_lambda)

n_bins = 25
all_bbox = pd.concat([df_base['BBox Ratio'], df_lambda['BBox Ratio']])
_, bin_edges = pd.cut(all_bbox, bins=n_bins, retbins=True)

df_base['BBox Bin'] = pd.cut(df_base['BBox Ratio'], bins=bin_edges, labels=False)
df_lambda['BBox Bin'] = pd.cut(df_lambda['BBox Ratio'], bins=bin_edges, labels=False)

pivot_base = df_base.pivot_table(values='Diff_Top5', index='BBox Bin', columns='Step', aggfunc='mean')
pivot_lambda = df_lambda.pivot_table(values='Diff_Top5', index='BBox Bin', columns='Step', aggfunc='mean')

step_values = sorted(set(pivot_base.columns) | set(pivot_lambda.columns))
pivot_base = pivot_base.reindex(columns=step_values)
pivot_lambda = pivot_lambda.reindex(columns=step_values)
pivot_diff = pivot_lambda - pivot_base

x_edges = np.concatenate([[0], step_values])
y_edges = bin_edges

fig, axes = plt.subplots(1, 3, figsize=(42, 10))

im0 = axes[0].pcolormesh(x_edges, y_edges, pivot_base.values, cmap='RdBu_r', vmin=-40, vmax=40)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Object Ratio')
axes[0].set_title(label_base)

im1 = axes[1].pcolormesh(x_edges, y_edges, pivot_lambda.values, cmap='RdBu_r', vmin=-40, vmax=40)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Object Ratio')
axes[1].set_title(label_lambda)
plt.colorbar(im1, ax=axes[0: 2], label='Top-5 Spuriosity')

diff_label = f"Diff ({label_lambda} - {label_base})"
im2 = axes[2].pcolormesh(x_edges, y_edges, pivot_diff.values, cmap='RdBu_r')
axes[2].set_xlabel('Step')
axes[2].set_ylabel('Object Ratio')
axes[2].set_title(diff_label)
plt.colorbar(im2, ax=axes[2], label='Top-5 Spuriosity Diff')

plt.show()

print(f"{label_base} - Total data points: {len(df_base)} (= ~50k samples x {len(step_values)} steps)")
print(f"{label_lambda} - Total data points: {len(df_lambda)} (= ~50k samples x {len(step_values)} steps)")
print(f"BBox Ratio range: {all_bbox.min():.4f} ~ {all_bbox.max():.4f}")

In [ ]:
target_epochs = range(len(steps))
epoch_names = {k: steps_int[k] for k in target_epochs}

def build_sample_df_conf(cache):
    """Build DataFrame using confidence-based metric instead of top-5 accuracy."""
    plot_data = []
    for epoch_idx in target_epochs:
        try:
            step_key = steps[epoch_idx]
        except (IndexError, KeyError):
            continue
        step_data = cache[step_key]
        step_value = epoch_names.get(epoch_idx, epoch_idx)
        for idx, data in step_data.items():
            cls = data['class']
            if cls not in class_data:
                continue
            sample_info = None
            for sample in class_data[cls]:
                if sample['idx'] == idx:
                    sample_info = sample
                    break
            bbox_ratio = sample_info['bbox_ratio']
            if bbox_ratio == 0.0:
                continue
            conf_bg = float(data['conf_bg'])
            conf_obj = float(data['conf_obj'])
            diff_conf = (conf_bg - conf_obj) * 100
            plot_data.append({
                'Step': step_value,
                'Class': cls,
                'Sample_Idx': idx,
                'Diff_Conf': diff_conf,
                'BBox Ratio': bbox_ratio
            })
    return pd.DataFrame(plot_data)

df_base = build_sample_df_conf(cache_base)
df_lambda = build_sample_df_conf(cache_lambda)

n_bins = 25
all_bbox = pd.concat([df_base['BBox Ratio'], df_lambda['BBox Ratio']])
_, bin_edges = pd.cut(all_bbox, bins=n_bins, retbins=True)

df_base['BBox Bin'] = pd.cut(df_base['BBox Ratio'], bins=bin_edges, labels=False)
df_lambda['BBox Bin'] = pd.cut(df_lambda['BBox Ratio'], bins=bin_edges, labels=False)

pivot_base = df_base.pivot_table(values='Diff_Conf', index='BBox Bin', columns='Step', aggfunc='mean')
pivot_lambda = df_lambda.pivot_table(values='Diff_Conf', index='BBox Bin', columns='Step', aggfunc='mean')

step_values = sorted(set(pivot_base.columns) | set(pivot_lambda.columns))
pivot_base = pivot_base.reindex(columns=step_values)
pivot_lambda = pivot_lambda.reindex(columns=step_values)
pivot_diff = pivot_lambda - pivot_base

x_edges = np.concatenate([[0], step_values])
y_edges = bin_edges

# Auto-detect vmin/vmax for the first two plots
vmin_main = min(pivot_base.min().min(), pivot_lambda.min().min())
vmax_main = max(pivot_base.max().max(), pivot_lambda.max().max())

# Symmetric range for diff plot
vabs_diff = max(abs(pivot_diff.min().min()), abs(pivot_diff.max().max()))

fig, axes = plt.subplots(1, 3, figsize=(42, 10))

im0 = axes[0].pcolormesh(x_edges, y_edges, pivot_base.values, cmap='RdBu_r', vmin=vmin_main, vmax=vmax_main)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Object Ratio')
axes[0].set_title(label_base)
plt.colorbar(im0, ax=axes[0], label='Confidence Spuriosity (conf_bg - conf_obj)')

im1 = axes[1].pcolormesh(x_edges, y_edges, pivot_lambda.values, cmap='RdBu_r', vmin=vmin_main, vmax=vmax_main)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Object Ratio')
axes[1].set_title(label_lambda)
plt.colorbar(im1, ax=axes[1], label='Confidence Spuriosity (conf_bg - conf_obj)')

diff_label = f"Diff ({label_lambda} - {label_base})"
im2 = axes[2].pcolormesh(x_edges, y_edges, pivot_diff.values, cmap='RdBu_r', vmin=-vabs_diff, vmax=vabs_diff)
axes[2].set_xlabel('Step')
axes[2].set_ylabel('Object Ratio')
axes[2].set_title(diff_label)
plt.colorbar(im2, ax=axes[2], label='Confidence Spuriosity Diff')

plt.tight_layout()
plt.show()

print(f"{label_base} - Total data points: {len(df_base)}")
print(f"{label_lambda} - Total data points: {len(df_lambda)}")
print(f"BBox Ratio range: {all_bbox.min():.4f} ~ {all_bbox.max():.4f}")
print(f"Conf diff range - base: [{pivot_base.min().min():.2f}, {pivot_base.max().max():.2f}]")
print(f"Conf diff range - lambda: [{pivot_lambda.min().min():.2f}, {pivot_lambda.max().max():.2f}]")
print(f"Diff range: [{pivot_diff.min().min():.4f}, {pivot_diff.max().max():.4f}]")

In [ ]:
check_step = "500"
lambda_configs = [
    ("lambda0.005", "/models/jhkim/imagenet500_lambda0.005"),
    ("lambda0.01",  "/models/jhkim/imagenet500_lambda0.01"),
    ("lambda0.05",  "/models/jhkim/imagenet500_lambda0.05"),
]

for name, path in lambda_configs:
    ckpt_path = f"{path}/{check_step}/checkpoint.pth"
    try:
        ckpt = torch.load(ckpt_path, weights_only=False, map_location='cpu')
        best_acc = ckpt.get('best_acc', 'N/A')
        epoch = ckpt.get('epoch', 'N/A')
        print(f"{name} (step {check_step}): best_acc={best_acc:.2f}%, trained_epochs={epoch}")
    except FileNotFoundError:
        print(f"{name} (step {check_step}): checkpoint not found at {ckpt_path}")
    except Exception as e:
        print(f"{name} (step {check_step}): error loading - {e}")